In [105]:
# Cell 1: Connect to TipQA SQL Server Database
import pandas as pd
import pyodbc
import os
from utilities.database_utils import get_sqlserver_connection
from utilities.shared_sync_utils import load_config

# Load configuration with environment variable substitution
print("Loading configuration...")
config = load_config()
print("Configuration loaded successfully")

# Connect to TipQA SQL Server database
print("Connecting to TipQA SQL Server database...")
conn = get_sqlserver_connection(config['tipqa_database'])
print(f"Connection established: {conn is not None}")


Loading configuration...
Configuration loaded successfully
Connecting to TipQA SQL Server database...
[INFO] Connecting to TipQA database: vSQLStd01.na.joby.aero:1433/TIPQA
[INFO] Using username: jae.osenbach
[INFO] Password set: Yes
[INFO] Successfully connected to TipQA database: vSQLStd01.na.joby.aero:1433/TIPQA in 0.23 seconds
Connection established: True


In [106]:
# Cell 2: Output TipQA tools data, tipqa_df
from utilities.database_utils import read_sql_query
import pandas as pd

# Read TipQA tools SQL query
print("Reading TipQA tools data...")
tipqa_sql = read_sql_query('tipqa_tools.sql')
tipqa_df = pd.read_sql(tipqa_sql, conn)

print(f"TipQA tools loaded: {len(tipqa_df):,} records")
print(f"Columns: {list(tipqa_df.columns)}")

# Convert service_interval_seconds to integer (remove decimal places)
if 'service_interval_seconds' in tipqa_df.columns:
    print("Converting service_interval_seconds to integer...")
    tipqa_df['service_interval_seconds'] = tipqa_df['service_interval_seconds'].fillna(0).astype(int)
    print("service_interval_seconds converted to integer")

# Standardize last_maintenance_date field
if 'last_maintenance_date' in tipqa_df.columns:
    print("Standardizing last_maintenance_date field...")
    # Convert to datetime, handling various formats
    tipqa_df['last_maintenance_date'] = pd.to_datetime(tipqa_df['last_maintenance_date'], errors='coerce')
    print("last_maintenance_date standardized to datetime format")

tipqa_df


Reading TipQA tools data...


/var/folders/2y/r__dx__n4k19hl9rr0vczgsc0000gp/T/ipykernel_86832/3033516132.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tipqa_df = pd.read_sql(tipqa_sql, conn)


TipQA tools loaded: 13,743 records
Columns: ['serial_number', 'part_number', 'description', 'revision', 'service_interval_seconds', 'asset_type', 'location', 'last_maintenance_date', 'asset_serial_number', 'manufacturer', 'maintenance_status', 'revision_status']
Converting service_interval_seconds to integer...
service_interval_seconds converted to integer
Standardizing last_maintenance_date field...
last_maintenance_date standardized to datetime format


,serial_number,part_number,description,revision,service_interval_seconds,asset_type,location,last_maintenance_date,asset_serial_number,manufacturer,maintenance_status,revision_status
0,J010-0073,2231A-30-3,Triple Output DC Power Supply,-,31557600,MTE,SRU02,NaT,8021960107610231,KEITHLEY,NEW,I
1,J010-0074,DP832A,Triple Output DC Power Supply,-,31557600,MTE,SRU02,NaT,DP8B194001107,RIGOL,NEW,I
2,J010-0075,*** UNKNOWN ***,"AIR FILTER, SOLDERING FA-430 ENDSAFE",-,31557600,MTE,SRU02,NaT,E212222,HAKKO,NEW,I
3,J010-0076,*** UNKNOWN ***,ELECTRONIC LOAD M9714,-,31557600,MTE,SRU02,NaT,None,MAYNUO ELECTRONICS,NEW,I
4,J010-0077,*** UNKNOWN ***,MS500-120,-,31557600,MTE,SRU02,NaT,1161-6262,MAGNA-POWER ELECTRONICS,NEW,I
...,...,...,...,...,...,...,...,...,...,...,...,...
13738,JE00006276,T510851-001-L4-003,"CMC, Battery Sleeve, Lamination Mold, 3m",A,0,EQP,MAK04,NaT,None,PROMMET,NEW,A
13739,JE00006277,T510851-001-L4-003,"CMC, Battery Sleeve, Lamination Mold, 3m",A,0,EQP,MAK04,NaT,None,PROMMET,NEW,A
13740,JE00006278,T510851-001-L4-003,"CMC, Battery Sleeve, Lamination Mold, 3m",A,0,EQP,MAK04,NaT,None,PROMMET,NEW,A
13741,JE00006279,T510851-001-L4-003,"CMC, Battery Sleeve, Lamination Mold, 3m",A,0,EQP,MAK04,NaT,None,PROMMET,NEW,A


In [ ]:
# Cell 3: Get Ion API token for V1 Production
from utilities.graphql_utils import get_token
import os

# Get Ion API token for V1 Production
print("Connecting to Ion API...")
environment = 'v1_production'

# Check if required environment variables are set
client_id = os.getenv('V1CLIENT')
client_secret = os.getenv('V1SECRET')
print(f"V1CLIENT environment variable: {'Set' if client_id else 'Not set'}")
print(f"V1SECRET environment variable: {'Set' if client_secret else 'Not set'}")

if not client_id or not client_secret:
    print("ERROR: Missing V1CLIENT or V1SECRET environment variables for v1_production")
    print("Please set these environment variables before running this cell")
else:
    token = get_token(config, environment=environment)
    print(f"Token obtained: {token is not None}")
    print(f"Environment: {environment}")


Connecting to Ion API...
CLIENT environment variable: Set
SECRET environment variable: Set
[INFO] Successfully obtained token for v1_production
Token obtained: True
Environment: v1_production


In [108]:
# Cell 4: Get Targeted Ion Inventory Data (by Serial/Part) and Create ion1_df DataFrame
from utilities.graphql_utils import post_graphql, read_query, get_token
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import pandas as pd
import os

print("Getting targeted Ion inventory data (by serial/part)...")
print("This will get BOTH partType=TOOL and partType=PART matches...")

# Check if token exists, if not get it
if 'token' not in locals() or token is None:
    print("Token not found, getting fresh token...")
    environment = 'v1_production'
    token = get_token(config, environment=environment)
    print(f"Token obtained: {token is not None}")
else:
    print("Using existing token")

# Step 1: Take all rows in TipQA output dataframe
print(f"Step 1: Total TipQA records: {len(tipqa_df):,}")

# Step 2: Choose only those rows that have serial numbers and part numbers
print("Step 2: Filtering for valid serial numbers and part numbers...")
valid_records = tipqa_df.dropna(subset=['serial_number', 'part_number']).copy()
valid_count = len(valid_records)
skipped_count = len(tipqa_df) - valid_count

print(f"Records with valid serial AND part numbers: {valid_count:,}")
print(f"Records skipped (missing serial or part number): {skipped_count:,}")

# Step 3: Prepare all valid records for Ion query processing
print("Step 3: Preparing records for Ion query processing...")
tipqa_records = []
for _, row in valid_records.iterrows():
    serial_number = str(row['serial_number']).strip()
    part_number = str(row['part_number']).strip()
    if serial_number and part_number:
        tipqa_records.append((serial_number, part_number))

print(f"Records prepared for Ion queries: {len(tipqa_records):,}")

# Read the targeted query that gets both TOOL and PART partTypes
targeted_query = read_query('get_tool_inventory_by_serial_and_part.graphql')
print(f"Query: {targeted_query[:200]}...")

def query_combination(serial_number, part_number):
    """Query a single serial/part combination and return all matching Ion data."""
    all_results = []
    after_cursor = None
    
    while True:
        variables = {
            "serialNumber": serial_number,
            "partNumber": part_number,
            "first": 100,
            "after": after_cursor
        }
        
        try:
            response = post_graphql(token, config, {'query': targeted_query, 'variables': variables}, environment)
            
            if 'errors' in response:
                # Only log errors for debugging, don't print every single error
                if len(all_results) == 0:  # Only log if no results found
                    print(f"GraphQL errors for {serial_number}/{part_number}: {response['errors']}")
                break
            
            edges = response.get('data', {}).get('partInventories', {}).get('edges', [])
            pageInfo = response.get('data', {}).get('partInventories', {}).get('pageInfo', {})
            
            # Collect results from this page
            for edge in edges:
                tool_node = edge.get('node', {})
                all_results.append(tool_node)
            
            # Check if there are more pages
            if not pageInfo.get('hasNextPage', False):
                break
                
            # Update cursor for next page
            after_cursor = pageInfo.get('endCursor')
            
        except Exception as e:
            print(f"Error querying {serial_number}/{part_number}: {e}")
            break
    
    return all_results

# Process all TipQA records in parallel with aggressive optimization
targeted_ion_data = []
start_time = time.time()

print(f"Step 4: Processing {len(tipqa_records)} TipQA records using aggressive parallel processing...")
print("This will query Ion for each TipQA record individually...")
print(f"With 100 workers, this should take ~{len(tipqa_records)/100/60:.1f} minutes (assuming ~1 second per API call)")

# Use ThreadPoolExecutor for parallel processing with aggressive settings
max_workers = min(100, len(tipqa_records))  # Use up to 100 workers
print(f"Using {max_workers} workers for maximum parallelization...")
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_combo = {
        executor.submit(query_combination, serial_number, part_number): (serial_number, part_number)
        for serial_number, part_number in tipqa_records
    }
    
    # Collect results as they complete with more frequent progress updates
    completed = 0
    for future in as_completed(future_to_combo):
        results = future.result()
        targeted_ion_data.extend(results)
        completed += 1
        
        # More frequent progress updates for better visibility
        if completed % 100 == 0 or completed == len(tipqa_records):
            elapsed = time.time() - start_time
            rate = completed / elapsed if elapsed > 0 else 0
            remaining = len(tipqa_records) - completed
            eta = remaining / rate if rate > 0 else 0
            print(f"Progress: {completed}/{len(tipqa_records)} records processed "
                  f"({completed/len(tipqa_records)*100:.1f}%). "
                  f"Rate: {rate:.1f} records/sec. ETA: {eta/60:.1f} minutes")

total_time = time.time() - start_time
print(f"\nCompleted processing {len(tipqa_records)} TipQA records in {total_time/60:.1f} minutes")
print(f"Targeted Ion data collected: {len(targeted_ion_data)} records")

# Create DataFrame from processed data
if targeted_ion_data:
    # Convert to DataFrame
    ion1_df = pd.DataFrame(targeted_ion_data)
    print(f"ion1_df created: {len(ion1_df)} rows, {len(ion1_df.columns)} columns")
    
    # Standardize null values - convert all None values to NaN for consistency
    print("Standardizing null values...")
    for col in ion1_df.columns:
        if ion1_df[col].dtype == 'object':  # Only process object columns that might contain None
            ion1_df[col] = ion1_df[col].replace({None: pd.NA})
    print("Null values standardized (None -> pd.NA)")
    
    # Standardize date fields
    print("Standardizing date fields...")
    date_columns = ['lastMaintainedDate']
    for col in date_columns:
        if col in ion1_df.columns:
            ion1_df[col] = pd.to_datetime(ion1_df[col], errors='coerce')
            print(f"Standardized {col} to datetime format")
    print("Date fields standardized")
    
    # Show partType distribution
    part_types = {}
    for record in targeted_ion_data:
        part_type = record.get('part', {}).get('partType', 'UNKNOWN')
        part_types[part_type] = part_types.get(part_type, 0) + 1
    
    print(f"PartType distribution: {part_types}")
    print("Data processing complete! DataFrame ready for review.")
    
    # Display the dataframe
    ion1_df
    
else:
    print("No data collected")
    ion1_df = pd.DataFrame()
    ion1_df


Getting targeted Ion inventory data (by serial/part)...
This will get BOTH partType=TOOL and partType=PART matches...
Using existing token
Step 1: Total TipQA records: 13,743
Step 2: Filtering for valid serial numbers and part numbers...
Records with valid serial AND part numbers: 13,513
Records skipped (missing serial or part number): 230
Step 3: Preparing records for Ion query processing...
Records prepared for Ion queries: 13,513
Query: query GetToolInventoryBySerialAndPart($first: Int, $after: String, $serialNumber: String!, $partNumber: String!) {
  partInventories(
    filters: {
      serialNumber: {eq: $serialNumber},
      part...
Step 4: Processing 13513 TipQA records using aggressive parallel processing...
This will query Ion for each TipQA record individually...
With 100 workers, this should take ~2.3 minutes (assuming ~1 second per API call)
Using 100 workers for maximum parallelization...
Progress: 100/13513 records processed (0.7%). Rate: 74.5 records/sec. ETA: 3.0 minut

In [109]:
# Cell 5: Flatten ion1_df to CSV-like structure
import pandas as pd
import json

print("Flattening ion1_df nested structures into individual columns...")

if 'ion1_df' in locals() and not ion1_df.empty:
    # Create a copy to work with
    flattened_df = ion1_df.copy()
    
    print(f"Original shape: {flattened_df.shape}")
    
    # 1. Expand location into location_id and location_name
    if 'location' in flattened_df.columns:
        print("Expanding location field...")
        location_data = flattened_df['location'].apply(lambda x: x if isinstance(x, dict) else {})
        flattened_df['location_id'] = location_data.apply(lambda x: x.get('id') if x else None)
        flattened_df['location_name'] = location_data.apply(lambda x: x.get('name') if x else None)
        
        # Convert location_id to integer (whole number, no decimals)
        print("Converting location_id to integer...")
        flattened_df['location_id'] = pd.to_numeric(flattened_df['location_id'], errors='coerce').astype('Int64')
        print("Location expanded into location_id and location_name")
    
    # 2. Expand attributes into individual columns (FIXED - attributes is a list of dicts)
    if 'attributes' in flattened_df.columns:
        print("Expanding attributes field...")
        attrs_data = flattened_df['attributes'].apply(lambda x: x if isinstance(x, list) else [])
        
        # Get all unique attribute keys from the list of dictionaries
        all_attr_keys = set()
        for attrs_list in attrs_data:
            if isinstance(attrs_list, list):
                for attr_dict in attrs_list:
                    if isinstance(attr_dict, dict) and 'key' in attr_dict:
                        all_attr_keys.add(attr_dict['key'])
        
        print(f"Found {len(all_attr_keys)} unique attribute keys: {list(all_attr_keys)}")
        
        # Create columns for each attribute key
        for key in all_attr_keys:
            col_name = f"attr_{key}"
            flattened_df[col_name] = attrs_data.apply(
                lambda x: next((attr_dict.get('value') for attr_dict in x if isinstance(attr_dict, dict) and attr_dict.get('key') == key), None)
                if isinstance(x, list) else None
            )
        print(f"Created {len(all_attr_keys)} attribute columns")
    
    # 3. Expand part into individual columns
    if 'part' in flattened_df.columns:
        print("Expanding part field...")
        part_data = flattened_df['part'].apply(lambda x: x if isinstance(x, dict) else {})
        
        # Get all unique part keys
        all_part_keys = set()
        for part in part_data:
            if isinstance(part, dict):
                all_part_keys.update(part.keys())
        
        print(f"Found {len(all_part_keys)} unique part keys: {list(all_part_keys)}")
        
        # Create columns for each part key
        for key in all_part_keys:
            col_name = f"part_{key}"
            flattened_df[col_name] = part_data.apply(lambda x: x.get(key) if x else None)
        print(f"Created {len(all_part_keys)} part columns")
        
        # Special handling for part_attributes - expand nested attributes
        # Check for part_attributes column (this is the flattened version from part expansion)
        if 'part_attributes' in flattened_df.columns:
            print("Expanding part_attributes field...")
            part_attrs_data = flattened_df['part_attributes'].apply(lambda x: x if isinstance(x, list) else [])
            
            # Get all unique part attribute keys
            all_part_attr_keys = set()
            for attrs_list in part_attrs_data:
                if isinstance(attrs_list, list):
                    for attr_dict in attrs_list:
                        if isinstance(attr_dict, dict) and 'key' in attr_dict:
                            all_part_attr_keys.add(attr_dict['key'])
            
            print(f"Found {len(all_part_attr_keys)} unique part attribute keys: {list(all_part_attr_keys)}")
            
            # Create columns for each part attribute key
            for key in all_part_attr_keys:
                col_name = f"part_attr_{key}"
                flattened_df[col_name] = part_attrs_data.apply(
                    lambda x: next((attr_dict.get('value') for attr_dict in x if isinstance(attr_dict, dict) and attr_dict.get('key') == key), None)
                    if isinstance(x, list) else None
                )
            print(f"Created {len(all_part_attr_keys)} part attribute columns")
            
            # Remove the original part_attributes column
            flattened_df = flattened_df.drop(columns=['part_attributes'])
            print("Removed original part_attributes column")
        
        # Convert part_maintenanceIntervalSeconds to integer (whole number, no decimals)
        if 'part_maintenanceIntervalSeconds' in flattened_df.columns:
            print("Converting part_maintenanceIntervalSeconds to integer...")
            flattened_df['part_maintenanceIntervalSeconds'] = pd.to_numeric(
                flattened_df['part_maintenanceIntervalSeconds'], errors='coerce'
            ).astype('Int64')
            print("part_maintenanceIntervalSeconds converted to integer")
    
    # 4. Expand abomInstallations
    if 'abomInstallations' in flattened_df.columns:
        print("Expanding abomInstallations field...")
        abom_data = flattened_df['abomInstallations'].apply(lambda x: x if isinstance(x, list) else [])
        
        # Get all unique abom installation keys
        all_abom_keys = set()
        for installations in abom_data:
            if isinstance(installations, list):
                for installation in installations:
                    if isinstance(installation, dict):
                        all_abom_keys.update(installation.keys())
        
        print(f"Found {len(all_abom_keys)} unique abom installation keys: {list(all_abom_keys)}")
        
        # Create columns for each abom installation key
        for key in all_abom_keys:
            col_name = f"abom_{key}"
            flattened_df[col_name] = abom_data.apply(
                lambda x: x[0].get(key) if isinstance(x, list) and len(x) > 0 and isinstance(x[0], dict) else None
            )
        print(f"Created {len(all_abom_keys)} abom installation columns")
        
        # Special handling for nested abom fields (like buildRequirement)
        for key in all_abom_keys:
            col_name = f"abom_{key}"
            if col_name in flattened_df.columns and flattened_df[col_name].dtype == 'object':
                # Check if this column contains dictionaries
                sample_values = flattened_df[col_name].dropna().head(5)
                if len(sample_values) > 0 and any(isinstance(x, dict) for x in sample_values):
                    print(f"Expanding nested abom_{key} field...")
                    nested_data = flattened_df[col_name].apply(lambda x: x if isinstance(x, dict) else {})
                    
                    # Get all unique nested keys
                    nested_keys = set()
                    for item in nested_data:
                        if isinstance(item, dict):
                            nested_keys.update(item.keys())
                    
                    # Create columns for each nested key
                    for nested_key in nested_keys:
                        nested_col_name = f"abom_{key}_{nested_key}"
                        flattened_df[nested_col_name] = nested_data.apply(lambda x: x.get(nested_key) if x else None)
                    
                    # Remove the original nested column
                    flattened_df = flattened_df.drop(columns=[col_name])
                    print(f"Expanded abom_{key} into {len(nested_keys)} nested columns")
    
    # Remove original nested columns (but keep the individual attribute columns we created)
    columns_to_remove = ['location', 'attributes', 'part', 'abomInstallations']
    columns_to_remove = [col for col in columns_to_remove if col in flattened_df.columns]
    flattened_df = flattened_df.drop(columns=columns_to_remove)
    
    print(f"Final shape: {flattened_df.shape}")
    print(f"Columns: {len(flattened_df.columns)}")
    
    # Update ion1_df with flattened version
    ion1_df = flattened_df
    
    print("ion1_df flattened successfully!")
    
    # Display the flattened dataframe
    ion1_df
    
else:
    print("No ion1_df found. Please run Cell 4 first.")

Flattening ion1_df nested structures into individual columns...
Original shape: (10653, 11)
Expanding location field...
Converting location_id to integer...
Location expanded into location_id and location_name
Expanding attributes field...
Found 19 unique attribute keys: ['Expiration Date', 'QMS Verified', 'Storage Location', 'DMD-###', 'ion_mirror_id', 'Pending Data', 'data', "counted - July '24", 'Routing Location', 'Joby PO', 'Quality Validation', 'Intended Aircraft', 'RI Number', 'QMS code', 'DAR Inspected', 'Intended Use', 'Usage Count', 'Manufacturer', 'Asset Serial Number']
Created 19 attribute columns
Expanding part field...
Found 9 unique part keys: ['attributes', '_etag', 'description', 'id', 'partNumber', 'trackingType', 'partType', 'revision', 'maintenanceIntervalSeconds']
Created 9 part columns
Expanding part_attributes field...
Found 7 unique part attribute keys: ['Asset Type', 'Primary Location', ' Used to Make', 'Blocks install with open issues', 'QMS Code', 'Manufactur

In [110]:
ion1_df

,id,serialNumber,_etag,status,unavailable,lotNumber,lastMaintainedDate,location_id,location_name,attr_Expiration Date,...,part_maintenanceIntervalSeconds,part_attr_Asset Type,part_attr_Primary Location,part_attr_ Used to Make,part_attr_Blocks install with open issues,part_attr_QMS Code,part_attr_Manufacturing Class,part_attr_Responsible ME,abom_id,abom_buildRequirement_id
0,786695,J010-0089,387100fe75754975be22402bb03d55f3,AVAILABLE,False,<NA>,NaT,<NA>,None,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
1,786740,J040-0023,11d843a6d56847c4a08be6783374d572,AVAILABLE,False,<NA>,NaT,<NA>,None,None,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
2,786730,J040-0018,7edd35fa6a4d46eb9959e903da5884e0,AVAILABLE,False,<NA>,NaT,<NA>,None,None,...,31536000,MTE,None,NaN,None,None,None,NaN,None,NaN
3,788121,J040-0053,a1cc880e02644f4c82ad0aace857c10a,AVAILABLE,False,<NA>,NaT,<NA>,None,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
4,786682,J010-0076,612fbaacfc2140c4992e041ab48ef732,AVAILABLE,False,<NA>,NaT,<NA>,None,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10648,794578,JE00006268,bcbd94db56a8486ca489e47ca7e59bc3,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
10649,794589,JE00006278,0643c851897a40c2b1373f1e16f876a0,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
10650,582706,JT00004860,32a364525b5749cd967150c7da75467e,AVAILABLE,False,<NA>,2024-11-21,2,MAK04,None,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
10651,595930,JE00004831,ec9c5816bf56400bb195592d3a7feb1d,AVAILABLE,False,<NA>,NaT,236,MAK02,None,...,<NA>,EQP,None,NaN,None,None,MechanicalAssembly,NaN,None,NaN


In [111]:
# Cell 6: Get all Ion tool inventory for V1 Production, output ion2_df (OPTIMIZED)
from utilities.graphql_utils import post_graphql, read_query
import time
import pandas as pd

print("Getting all Ion inventory data using optimized processing...")

# Read the all inventory query
all_query = read_query('get_all_tool_inventory.graphql')
print(f"Query: {all_query[:200]}...")

def get_all_ion_data_optimized():
    """Get all Ion inventory data using optimized pagination."""
    all_data = []
    after_cursor = None
    page_count = 0
    
    print("Starting optimized data collection...")
    start_time = time.time()
    
    while True:
        variables = {
            "first": 1000,  # Larger page size for efficiency
            "after": after_cursor
        }
        
        try:
            response = post_graphql(token, config, {'query': all_query, 'variables': variables}, environment)
            
            if 'errors' in response:
                print(f"GraphQL errors: {response['errors']}")
                break
            
            edges = response.get('data', {}).get('partInventories', {}).get('edges', [])
            pageInfo = response.get('data', {}).get('partInventories', {}).get('pageInfo', {})
            
            # Collect results from this page
            page_data = []
            for edge in edges:
                tool_node = edge.get('node', {})
                page_data.append(tool_node)
            
            all_data.extend(page_data)
            page_count += 1
            
            # Progress update
            elapsed = time.time() - start_time
            rate = len(all_data) / elapsed if elapsed > 0 else 0
            print(f"Page {page_count}: Collected {len(page_data)} records. "
                  f"Total: {len(all_data)} records. Rate: {rate:.1f} records/sec")
            
            # Check if there are more pages
            if not pageInfo.get('hasNextPage', False):
                break
                
            # Update cursor for next page
            after_cursor = pageInfo.get('endCursor')
            
        except Exception as e:
            print(f"Error on page {page_count + 1}: {e}")
            break
    
    total_time = time.time() - start_time
    print(f"\nCompleted collecting {len(all_data)} records in {total_time/60:.1f} minutes")
    return all_data

# Get all Ion tool inventory using optimized method
all_ion_data = get_all_ion_data_optimized()

print(f"All Ion data collected: {len(all_ion_data)} records")
if all_ion_data:
    print(f"Sample record keys: {list(all_ion_data[0].keys())}")
    
    # Convert to DataFrame for easier analysis
    ion2_df = pd.DataFrame(all_ion_data)
    print(f"\nion2_df created: {len(ion2_df)} rows, {len(ion2_df.columns)} columns")
    print(f"Columns: {list(ion2_df.columns)}")
    
    # Standardize null values - convert all None values to NaN for consistency
    print("Standardizing null values...")
    print(f"DataFrame shape before: {ion2_df.shape}")
    
    # More efficient approach: only process columns that might have None values
    # Convert None to NaN more efficiently
    for col in ion2_df.columns:
        if ion2_df[col].dtype == 'object':  # Only process object columns that might contain None
            ion2_df[col] = ion2_df[col].replace({None: pd.NA})
    
    print("Null values standardized (None -> pd.NA)")
    
    # Standardize date fields
    print("Standardizing date fields...")
    date_columns = ['lastMaintainedDate']
    for col in date_columns:
        if col in ion2_df.columns:
            ion2_df[col] = pd.to_datetime(ion2_df[col], errors='coerce')
            print(f"Standardized {col} to datetime format")
    
    print("Date fields standardized")
    
    # Show the DataFrame using print
    print(f"\nion2_df (first 5 rows):")
    print(ion2_df.head(5))
    
    # Show DataFrame info
    print(f"\nDataFrame info:")
    print(f"Shape: {ion2_df.shape}")
    
    # Display the dataframe
    ion2_df
    
else:
    print("No data collected")
    ion2_df = pd.DataFrame()
    ion2_df

Getting all Ion inventory data using optimized processing...
Query: query GetAllToolInventory($first: Int, $after: String) {
  partInventories(
    filters: {
      part: {
        partType: {eq: TOOL}
      }
    }
    first: $first
    after: $after
  ) {
    edges ...
Starting optimized data collection...
Page 1: Collected 1000 records. Total: 1000 records. Rate: 274.2 records/sec
Page 2: Collected 1000 records. Total: 2000 records. Rate: 276.7 records/sec
Page 3: Collected 1000 records. Total: 3000 records. Rate: 261.0 records/sec
Page 4: Collected 1000 records. Total: 4000 records. Rate: 252.6 records/sec
Page 5: Collected 1000 records. Total: 5000 records. Rate: 249.0 records/sec
Page 6: Collected 1000 records. Total: 6000 records. Rate: 249.5 records/sec
Page 7: Collected 1000 records. Total: 7000 records. Rate: 243.1 records/sec
Page 8: Collected 1000 records. Total: 8000 records. Rate: 238.3 records/sec
Page 9: Collected 1000 records. Total: 9000 records. Rate: 237.7 records/s

In [112]:
# Cell 8: Verify ion1_df and ion2_df have the same width
import pandas as pd

print("=== DATAFRAME WIDTH COMPARISON ===")

# Check if both dataframes exist
if 'ion1_df' in locals() and 'ion2_df' in locals():
    print(f"ion1_df shape: {ion1_df.shape}")
    print(f"ion2_df shape: {ion2_df.shape}")
    
    ion1_width = len(ion1_df.columns)
    ion2_width = len(ion2_df.columns)
    
    print(f"\nion1_df columns: {ion1_width}")
    print(f"ion2_df columns: {ion2_width}")
    
    if ion1_width == ion2_width:
        print("SUCCESS: Both dataframes have the same width!")
        print(f"Both have {ion1_width} columns")
    else:
        print("MISMATCH: Dataframes have different widths")
        print(f"Difference: {abs(ion1_width - ion2_width)} columns")
        
        # Show which columns are different
        ion1_cols = set(ion1_df.columns)
        ion2_cols = set(ion2_df.columns)
        
        only_in_ion1 = ion1_cols - ion2_cols
        only_in_ion2 = ion2_cols - ion1_cols
        
        if only_in_ion1:
            print(f"\nColumns only in ion1_df ({len(only_in_ion1)}):")
            for col in sorted(only_in_ion1):
                print(f"  - {col}")
        
        if only_in_ion2:
            print(f"\nColumns only in ion2_df ({len(only_in_ion2)}):")
            for col in sorted(only_in_ion2):
                print(f"  - {col}")
    
    print(f"\n=== COLUMN BREAKDOWN ===")
    print(f"ion1_df columns:")
    for i, col in enumerate(sorted(ion1_df.columns), 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\nion2_df columns:")
    for i, col in enumerate(sorted(ion2_df.columns), 1):
        print(f"  {i:2d}. {col}")
    
    # Check for common columns
    common_cols = ion1_cols & ion2_cols
    print(f"\nCommon columns: {len(common_cols)}")
    print(f"Common column names: {sorted(common_cols)}")
    
else:
    print("ERROR: One or both dataframes are missing")
    if 'ion1_df' not in locals():
        print("  - ion1_df is missing")
    if 'ion2_df' not in locals():
        print("  - ion2_df is missing")
    print("\nPlease run the previous cells to create both dataframes first.")


=== DATAFRAME WIDTH COMPARISON ===
ion1_df shape: (10653, 45)
ion2_df shape: (12750, 11)

ion1_df columns: 45
ion2_df columns: 11
MISMATCH: Dataframes have different widths
Difference: 34 columns

Columns only in ion1_df (38):
  - abom_buildRequirement_id
  - abom_id
  - attr_Asset Serial Number
  - attr_DAR Inspected
  - attr_DMD-###
  - attr_Expiration Date
  - attr_Intended Aircraft
  - attr_Intended Use
  - attr_Joby PO
  - attr_Manufacturer
  - attr_Pending Data
  - attr_QMS Verified
  - attr_QMS code
  - attr_Quality Validation
  - attr_RI Number
  - attr_Routing Location
  - attr_Storage Location
  - attr_Usage Count
  - attr_counted - July '24
  - attr_data
  - attr_ion_mirror_id
  - location_id
  - location_name
  - part__etag
  - part_attr_ Used to Make
  - part_attr_Asset Type
  - part_attr_Blocks install with open issues
  - part_attr_Manufacturing Class
  - part_attr_Primary Location
  - part_attr_QMS Code
  - part_attr_Responsible ME
  - part_description
  - part_id
  - p

In [113]:
ion2_df

,id,serialNumber,_etag,status,unavailable,lotNumber,lastMaintainedDate,location,attributes,part,abomInstallations
0,11759,JE00002430,e70f28714053452895e1a2da1d435ca4,AVAILABLE,False,<NA>,NaT,<NA>,"[{'key': 'Asset Serial Number', 'value': None,...","{'id': 1951, 'partNumber': 'T200232-001-L1', '...",[]
1,18173,JE00000755,f71d9832988b4e2db5db8ab855ab6303,AVAILABLE,False,<NA>,NaT,"{'id': 236, 'name': 'MAK02'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 2306, 'partNumber': 'T203172-001-M1-102...",[]
2,18174,JE00000757,c2ad2c161725401984fb113daac58ad3,AVAILABLE,False,<NA>,NaT,"{'id': 236, 'name': 'MAK02'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 2306, 'partNumber': 'T203172-001-M1-102...",[]
3,18175,JE0000758,33f9f1422f3b42b981e9bf83589b3f03,UNAVAILABLE,True,<NA>,NaT,"{'id': 12360, 'name': 'LOST - Lost/Obsolete Eq...","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 2306, 'partNumber': 'T203172-001-M1-102...",[]
4,18176,JE00000756,02af1d35167943b3bb781a9aef3aa731,AVAILABLE,False,<NA>,NaT,"{'id': 236, 'name': 'MAK02'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 2306, 'partNumber': 'T203172-001-M1-102...",[]
...,...,...,...,...,...,...,...,...,...,...,...
12745,794587,JE00006277,aa10c89ef7f443118192f068f334a57a,AVAILABLE,False,<NA>,NaT,"{'id': 2, 'name': 'MAK04'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 60158, 'partNumber': 'T510851-001-L4-00...",[]
12746,794588,JE00006275,673061fa45284e2489298707d43a60ea,AVAILABLE,False,<NA>,NaT,"{'id': 2, 'name': 'MAK04'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 60158, 'partNumber': 'T510851-001-L4-00...",[]
12747,794589,JE00006278,0643c851897a40c2b1373f1e16f876a0,AVAILABLE,False,<NA>,NaT,"{'id': 2, 'name': 'MAK04'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 60158, 'partNumber': 'T510851-001-L4-00...",[]
12748,794590,JE00006279,cae2e5dfc4fa45f48ecad4003b14a346,AVAILABLE,False,<NA>,NaT,"{'id': 2, 'name': 'MAK04'}","[{'key': 'Asset Serial Number', 'value': None,...","{'id': 60158, 'partNumber': 'T510851-001-L4-00...",[]


In [114]:
# Cell 7: Flatten ion2_df to CSV-like structure
import pandas as pd

print("Flattening ion2_df nested structures into individual columns...")
print(f"Original shape: {ion2_df.shape}")

# Start with a copy of the original dataframe
ion2_df = ion2_df.copy()

# 1. Expand location field
if 'location' in ion2_df.columns:
    print("Expanding location field...")
    ion2_df['location_id'] = ion2_df['location'].apply(lambda x: x.get('id') if isinstance(x, dict) else None)
    ion2_df['location_name'] = ion2_df['location'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)
    
    # Convert location_id to integer (whole number, no decimals)
    print("Converting location_id to integer...")
    ion2_df['location_id'] = pd.to_numeric(ion2_df['location_id'], errors='coerce').astype('Int64')
    
    print("Location expanded into location_id and location_name")

# 2. Expand attributes field
if 'attributes' in ion2_df.columns:
    print("Expanding attributes field...")
    attrs_data = ion2_df['attributes'].apply(lambda x: x if isinstance(x, list) else [])
    
    all_attr_keys = set()
    for attrs_list in attrs_data:
        if isinstance(attrs_list, list):
            for attr_dict in attrs_list:
                if isinstance(attr_dict, dict) and 'key' in attr_dict:
                    all_attr_keys.add(attr_dict['key'])
    
    print(f"Found {len(all_attr_keys)} unique attribute keys: {list(all_attr_keys)}")
    
    for key in all_attr_keys:
        col_name = f"attr_{key}"
        ion2_df[col_name] = attrs_data.apply(
            lambda x: next((attr_dict.get('value') for attr_dict in x if isinstance(attr_dict, dict) and attr_dict.get('key') == key), None)
            if isinstance(x, list) else None
        )
    print(f"Created {len(all_attr_keys)} attribute columns")

# 3. Expand part field
if 'part' in ion2_df.columns:
    print("Expanding part field...")
    part_data = ion2_df['part'].apply(lambda x: x if isinstance(x, dict) else {})
    
    all_part_keys = set()
    for part_dict in part_data:
        if isinstance(part_dict, dict):
            all_part_keys.update(part_dict.keys())
    
    print(f"Found {len(all_part_keys)} unique part keys: {list(all_part_keys)}")
    
    for key in all_part_keys:
        col_name = f"part_{key}"
        ion2_df[col_name] = part_data.apply(lambda x: x.get(key) if isinstance(x, dict) else None)
    print(f"Created {len(all_part_keys)} part columns")

# 4. Expand part_attributes (from part expansion)
if 'part_attributes' in ion2_df.columns:
    print("Expanding part_attributes field...")
    part_attrs_data = ion2_df['part_attributes'].apply(lambda x: x if isinstance(x, list) else [])
    
    all_part_attr_keys = set()
    for attrs_list in part_attrs_data:
        if isinstance(attrs_list, list):
            for attr_dict in attrs_list:
                if isinstance(attr_dict, dict) and 'key' in attr_dict:
                    all_part_attr_keys.add(attr_dict['key'])
    
    print(f"Found {len(all_part_attr_keys)} unique part attribute keys: {list(all_part_attr_keys)}")
    
    for key in all_part_attr_keys:
        col_name = f"part_attr_{key}"
        ion2_df[col_name] = part_attrs_data.apply(
            lambda x: next((attr_dict.get('value') for attr_dict in x if isinstance(attr_dict, dict) and attr_dict.get('key') == key), None)
            if isinstance(x, list) else None
        )
    print(f"Created {len(all_part_attr_keys)} part attribute columns")
    
    # Remove the original part_attributes column
    ion2_df = ion2_df.drop(columns=['part_attributes'])
    print("Removed original part_attributes column")

# 5. Convert part_maintenanceIntervalSeconds to integer (whole number, no decimals)
if 'part_maintenanceIntervalSeconds' in ion2_df.columns:
    print("Converting part_maintenanceIntervalSeconds to integer...")
    ion2_df['part_maintenanceIntervalSeconds'] = pd.to_numeric(
        ion2_df['part_maintenanceIntervalSeconds'], errors='coerce'
    ).astype('Int64')
    print("part_maintenanceIntervalSeconds converted to integer")

# 6. Expand abomInstallations
if 'abomInstallations' in ion2_df.columns:
    print("Expanding abomInstallations field...")
    abom_data = ion2_df['abomInstallations'].apply(lambda x: x if isinstance(x, list) else [])
    
    all_abom_keys = set()
    for installations in abom_data:
        if isinstance(installations, list):
            for installation in installations:
                if isinstance(installation, dict):
                    all_abom_keys.update(installation.keys())
    
    print(f"Found {len(all_abom_keys)} unique abom installation keys: {list(all_abom_keys)}")
    
    for key in all_abom_keys:
        col_name = f"abom_{key}"
        ion2_df[col_name] = abom_data.apply(
            lambda x: x[0].get(key) if isinstance(x, list) and len(x) > 0 and isinstance(x[0], dict) else None
        )
    print(f"Created {len(all_abom_keys)} abom installation columns")
    
    # Special handling for nested abom fields (like buildRequirement)
    for key in all_abom_keys:
        col_name = f"abom_{key}"
        if col_name in ion2_df.columns and ion2_df[col_name].dtype == 'object':
            sample_values = ion2_df[col_name].dropna().head(5)
            if len(sample_values) > 0 and any(isinstance(x, dict) for x in sample_values):
                print(f"Expanding nested abom_{key} field...")
                nested_data = ion2_df[col_name].apply(lambda x: x if isinstance(x, dict) else {})
                
                nested_keys = set()
                for item in nested_data:
                    if isinstance(item, dict):
                        nested_keys.update(item.keys())
                
                for nested_key in nested_keys:
                    nested_col_name = f"abom_{key}_{nested_key}"
                    ion2_df[nested_col_name] = nested_data.apply(lambda x: x.get(nested_key) if x else None)
                
                ion2_df = ion2_df.drop(columns=[col_name])
                print(f"Expanded abom_{key} into {len(nested_keys)} nested columns")

# Remove original nested columns
columns_to_remove = ['location', 'attributes', 'part', 'abomInstallations']
columns_to_remove = [col for col in columns_to_remove if col in ion2_df.columns]
ion2_df = ion2_df.drop(columns=columns_to_remove)

print(f"\nFlattening complete!")
print(f"Final shape: {ion2_df.shape}")
print(f"Columns: {len(ion2_df.columns)}")

# Show column information
print(f"\nColumn breakdown:")
print(f"Original columns: {len(ion2_df.columns)}")
print(f"Flattened columns: {len(ion2_df.columns)}")
print(f"New columns added: {len(columns_to_remove)}")

# Display the flattened dataframe
ion2_df

Flattening ion2_df nested structures into individual columns...
Original shape: (12750, 11)
Expanding location field...
Converting location_id to integer...
Location expanded into location_id and location_name
Expanding attributes field...
Found 19 unique attribute keys: ['Expiration Date', 'QMS Verified', 'Storage Location', 'DMD-###', 'ion_mirror_id', 'Pending Data', 'data', "counted - July '24", 'Routing Location', 'Joby PO', 'Quality Validation', 'Intended Aircraft', 'RI Number', 'QMS code', 'DAR Inspected', 'Intended Use', 'Usage Count', 'Manufacturer', 'Asset Serial Number']
Created 19 attribute columns
Expanding part field...
Found 9 unique part keys: ['attributes', '_etag', 'description', 'id', 'partNumber', 'trackingType', 'partType', 'revision', 'maintenanceIntervalSeconds']
Created 9 part columns
Expanding part_attributes field...
Found 7 unique part attribute keys: ['Asset Type', 'Primary Location', ' Used to Make', 'Blocks install with open issues', 'QMS Code', 'Manufactur

,id,serialNumber,_etag,status,unavailable,lotNumber,lastMaintainedDate,location_id,location_name,attr_Expiration Date,...,part_maintenanceIntervalSeconds,part_attr_Asset Type,part_attr_Primary Location,part_attr_ Used to Make,part_attr_Blocks install with open issues,part_attr_QMS Code,part_attr_Manufacturing Class,part_attr_Responsible ME,abom_id,abom_buildRequirement_id
0,11759,JE00002430,e70f28714053452895e1a2da1d435ca4,AVAILABLE,False,<NA>,NaT,<NA>,None,None,...,<NA>,EQP,repair,NaN,None,NONCONF,None,NaN,None,NaN
1,18173,JE00000755,f71d9832988b4e2db5db8ab855ab6303,AVAILABLE,False,<NA>,NaT,236,MAK02,None,...,<NA>,EQP,MAK06,NaN,None,None,None,NaN,None,NaN
2,18174,JE00000757,c2ad2c161725401984fb113daac58ad3,AVAILABLE,False,<NA>,NaT,236,MAK02,None,...,<NA>,EQP,MAK06,NaN,None,None,None,NaN,None,NaN
3,18175,JE0000758,33f9f1422f3b42b981e9bf83589b3f03,UNAVAILABLE,True,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,EQP,MAK06,NaN,None,None,None,NaN,None,NaN
4,18176,JE00000756,02af1d35167943b3bb781a9aef3aa731,AVAILABLE,False,<NA>,NaT,236,MAK02,None,...,<NA>,EQP,MAK06,NaN,None,None,None,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12745,794587,JE00006277,aa10c89ef7f443118192f068f334a57a,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
12746,794588,JE00006275,673061fa45284e2489298707d43a60ea,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
12747,794589,JE00006278,0643c851897a40c2b1373f1e16f876a0,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
12748,794590,JE00006279,cae2e5dfc4fa45f48ecad4003b14a346,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN


In [115]:
# Cell 8: Verify ion1_df and ion2_df have the same width
import pandas as pd

print("=== DATAFRAME WIDTH COMPARISON ===")

# Check if both dataframes exist
if 'ion1_df' in locals() and 'ion2_df' in locals():
    print(f"ion1_df shape: {ion1_df.shape}")
    print(f"ion2_df shape: {ion2_df.shape}")
    
    ion1_width = len(ion1_df.columns)
    ion2_width = len(ion2_df.columns)
    
    print(f"\nion1_df columns: {ion1_width}")
    print(f"ion2_df columns: {ion2_width}")
    
    if ion1_width == ion2_width:
        print("SUCCESS: Both dataframes have the same width!")
        print(f"Both have {ion1_width} columns")
    else:
        print("MISMATCH: Dataframes have different widths")
        print(f"Difference: {abs(ion1_width - ion2_width)} columns")
        
        # Show which columns are different
        ion1_cols = set(ion1_df.columns)
        ion2_cols = set(ion2_df.columns)
        
        only_in_ion1 = ion1_cols - ion2_cols
        only_in_ion2 = ion2_cols - ion1_cols
        
        if only_in_ion1:
            print(f"\nColumns only in ion1_df ({len(only_in_ion1)}):")
            for col in sorted(only_in_ion1):
                print(f"  - {col}")
        
        if only_in_ion2:
            print(f"\nColumns only in ion2_df ({len(only_in_ion2)}):")
            for col in sorted(only_in_ion2):
                print(f"  - {col}")
    
    print(f"\n=== COLUMN BREAKDOWN ===")
    print(f"ion1_df columns:")
    for i, col in enumerate(sorted(ion1_df.columns), 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\nion2_df columns:")
    for i, col in enumerate(sorted(ion2_df.columns), 1):
        print(f"  {i:2d}. {col}")
    
    # Check for common columns
    common_cols = ion1_cols & ion2_cols
    print(f"\nCommon columns: {len(common_cols)}")
    print(f"Common column names: {sorted(common_cols)}")
    
else:
    print("ERROR: One or both dataframes are missing")
    if 'ion1_df' not in locals():
        print("  - ion1_df is missing")
    if 'ion2_df' not in locals():
        print("  - ion2_df is missing")
    print("\nPlease run the previous cells to create both dataframes first.")


=== DATAFRAME WIDTH COMPARISON ===
ion1_df shape: (10653, 45)
ion2_df shape: (12750, 45)

ion1_df columns: 45
ion2_df columns: 45
SUCCESS: Both dataframes have the same width!
Both have 45 columns

=== COLUMN BREAKDOWN ===
ion1_df columns:
   1. _etag
   2. abom_buildRequirement_id
   3. abom_id
   4. attr_Asset Serial Number
   5. attr_DAR Inspected
   6. attr_DMD-###
   7. attr_Expiration Date
   8. attr_Intended Aircraft
   9. attr_Intended Use
  10. attr_Joby PO
  11. attr_Manufacturer
  12. attr_Pending Data
  13. attr_QMS Verified
  14. attr_QMS code
  15. attr_Quality Validation
  16. attr_RI Number
  17. attr_Routing Location
  18. attr_Storage Location
  19. attr_Usage Count
  20. attr_counted - July '24
  21. attr_data
  22. attr_ion_mirror_id
  23. id
  24. lastMaintainedDate
  25. location_id
  26. location_name
  27. lotNumber
  28. part__etag
  29. part_attr_ Used to Make
  30. part_attr_Asset Type
  31. part_attr_Blocks install with open issues
  32. part_attr_Manufactur

In [116]:
# Cell 9: Append ion1_df and ion2_df together
import pandas as pd

print("=== APPENDING ION DATAFRAMES ===")

# Check if both dataframes exist
if 'ion1_df' in locals() and 'ion2_df' in locals():
    print(f"ion1_df shape: {ion1_df.shape}")
    print(f"ion2_df shape: {ion2_df.shape}")
    
    # Check if they have the same columns
    ion1_cols = set(ion1_df.columns)
    ion2_cols = set(ion2_df.columns)
    
    if ion1_cols == ion2_cols:
        print("Both dataframes have the same columns - safe to append")
        
        # Append the dataframes
        combined_df = pd.concat([ion1_df, ion2_df], ignore_index=True)
        
        print(f"Combined dataframe shape: {combined_df.shape}")
        print(f"Total records: {len(combined_df):,}")
        
        # Show breakdown
        print(f"\nBreakdown:")
        print(f"  - ion1_df records: {len(ion1_df):,}")
        print(f"  - ion2_df records: {len(ion2_df):,}")
        print(f"  - Combined total: {len(combined_df):,}")
        
        # Check for any duplicates (optional)
        print(f"\nChecking for duplicates...")
        duplicate_count = combined_df.duplicated().sum()
        print(f"Duplicate records found: {duplicate_count:,}")
        
        if duplicate_count > 0:
            print("Duplicates detected - you may want to remove them")
            print("To remove duplicates, run: combined_df = combined_df.drop_duplicates()")
        
        # Display the combined dataframe
        combined_df
        
    else:
        print("Dataframes have different columns - cannot append safely")
        print(f"Columns only in ion1_df: {ion1_cols - ion2_cols}")
        print(f"Columns only in ion2_df: {ion2_cols - ion1_cols}")
        
else:
    print("One or both dataframes are missing")
    if 'ion1_df' not in locals():
        print("  - ion1_df is missing")
    if 'ion2_df' not in locals():
        print("  - ion2_df is missing")
    print("\nPlease run the previous cells to create both dataframes first.")

=== APPENDING ION DATAFRAMES ===
ion1_df shape: (10653, 45)
ion2_df shape: (12750, 45)
Both dataframes have the same columns - safe to append
Combined dataframe shape: (23403, 45)
Total records: 23,403

Breakdown:
  - ion1_df records: 10,653
  - ion2_df records: 12,750
  - Combined total: 23,403

Checking for duplicates...
Duplicate records found: 10,608
Duplicates detected - you may want to remove them
To remove duplicates, run: combined_df = combined_df.drop_duplicates()


/var/folders/2y/r__dx__n4k19hl9rr0vczgsc0000gp/T/ipykernel_86832/2456665851.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([ion1_df, ion2_df], ignore_index=True)


In [117]:
combined_df.sort_values(by='serialNumber', inplace=True, na_position='last')
combined_df

,id,serialNumber,_etag,status,unavailable,lotNumber,lastMaintainedDate,location_id,location_name,attr_Expiration Date,...,part_maintenanceIntervalSeconds,part_attr_Asset Type,part_attr_Primary Location,part_attr_ Used to Make,part_attr_Blocks install with open issues,part_attr_QMS Code,part_attr_Manufacturing Class,part_attr_Responsible ME,abom_id,abom_buildRequirement_id
15254,489394,\nJE00003550,5f7b3fe134754ab09f102ff1681f42c2,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
14240,244462,JE00000378 - DO NOT USE,6e1ec6462887417d9e0d8e5aa11b2c14,UNAVAILABLE,False,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
10676,18200,JE00000663,05e377cfe3d04b45b777a37d091de45e,UNAVAILABLE,True,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
13438,209568,JE00002626 - DO NOT USE,74c24bc377534e908dfee690ee3c04dd,UNAVAILABLE,False,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,EQP,None,NaN,None,None,Machined,NaN,None,NaN
15104,477119,JE00003867,aaa3a51b622c431ba95525322a846a7a,AVAILABLE,False,<NA>,NaT,4972,MAK06,None,...,0,EQP,None,NaN,None,NONCONF,MechanicalAssembly,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10733,22224,placeholder,c4def01121284b82839c84731997b460,AVAILABLE,True,<NA>,NaT,258,ESI101010A,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
10734,22225,placeholder1,53402acb6f9849c08350ac590122b022,AVAILABLE,True,<NA>,NaT,258,ESI101010A,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
15500,516439,JT00004484,4d8aab6179f148008f9be77fcc46d2e0,AVAILABLE,False,<NA>,2024-10-19 07:00:00,1733,SQL01,None,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
15509,516451,JT00004485,a96008102bab43eda6906c4d082f4b26,AVAILABLE,False,<NA>,2024-10-19 07:00:00,1733,SQL01,None,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN


In [118]:
# Remove duplicates from combined dataframe
print("=== REMOVING DUPLICATES ===")

print(f"Original combined_df shape: {combined_df.shape}")
print(f"Original record count: {len(combined_df):,}")

# Check for duplicates
duplicate_count = combined_df.duplicated().sum()
print(f"Duplicate records found: {duplicate_count:,}")

if duplicate_count > 0:
    # Remove duplicates
    combined_df1 = combined_df.drop_duplicates()
    
    print(f"After removing duplicates:")
    print(f"New shape: {combined_df1.shape}")
    print(f"New record count: {len(combined_df1):,}")
    print(f"Records removed: {duplicate_count:,}")
    
    # Display the deduplicated dataframe
    combined_df1
    
else:
    print("No duplicates found - dataframe unchanged")
    combined_df

=== REMOVING DUPLICATES ===
Original combined_df shape: (23403, 45)
Original record count: 23,403
Duplicate records found: 10,608
After removing duplicates:
New shape: (12795, 45)
New record count: 12,795
Records removed: 10,608


In [119]:
combined_df1

,id,serialNumber,_etag,status,unavailable,lotNumber,lastMaintainedDate,location_id,location_name,attr_Expiration Date,...,part_maintenanceIntervalSeconds,part_attr_Asset Type,part_attr_Primary Location,part_attr_ Used to Make,part_attr_Blocks install with open issues,part_attr_QMS Code,part_attr_Manufacturing Class,part_attr_Responsible ME,abom_id,abom_buildRequirement_id
15254,489394,\nJE00003550,5f7b3fe134754ab09f102ff1681f42c2,AVAILABLE,False,<NA>,NaT,2,MAK04,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
14240,244462,JE00000378 - DO NOT USE,6e1ec6462887417d9e0d8e5aa11b2c14,UNAVAILABLE,False,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
10676,18200,JE00000663,05e377cfe3d04b45b777a37d091de45e,UNAVAILABLE,True,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,EQP,None,NaN,None,None,None,NaN,None,NaN
13438,209568,JE00002626 - DO NOT USE,74c24bc377534e908dfee690ee3c04dd,UNAVAILABLE,False,<NA>,NaT,12360,LOST - Lost/Obsolete Equipment,None,...,<NA>,EQP,None,NaN,None,None,Machined,NaN,None,NaN
15104,477119,JE00003867,aaa3a51b622c431ba95525322a846a7a,AVAILABLE,False,<NA>,NaT,4972,MAK06,None,...,0,EQP,None,NaN,None,NONCONF,MechanicalAssembly,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10733,22224,placeholder,c4def01121284b82839c84731997b460,AVAILABLE,True,<NA>,NaT,258,ESI101010A,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
10734,22225,placeholder1,53402acb6f9849c08350ac590122b022,AVAILABLE,True,<NA>,NaT,258,ESI101010A,None,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
15500,516439,JT00004484,4d8aab6179f148008f9be77fcc46d2e0,AVAILABLE,False,<NA>,2024-10-19 07:00:00,1733,SQL01,None,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
15509,516451,JT00004485,a96008102bab43eda6906c4d082f4b26,AVAILABLE,False,<NA>,2024-10-19 07:00:00,1733,SQL01,None,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN


In [120]:
# Append tipqa_df and combined_df1 with column renaming and matching
import pandas as pd

print("=== APPENDING TIPQA AND ION DATAFRAMES ===")

# Check if both dataframes exist
if 'tipqa_df' in locals() and 'combined_df1' in locals():
    print(f"tipqa_df shape: {tipqa_df.shape}")
    print(f"combined_df1 shape: {combined_df1.shape}")
    
    # Create copies to work with
    tipqa_renamed = tipqa_df.copy()
    ion_renamed = combined_df1.copy()
    
    # Rename tipqa columns with tipqa_ prefix
    print("Renaming tipqa columns...")
    tipqa_renamed.columns = ['tipqa_' + col for col in tipqa_renamed.columns]
    print(f"TipQA columns renamed: {list(tipqa_renamed.columns)}")
    
    # Rename ion columns with ion_ prefix
    print("Renaming ion columns...")
    ion_renamed.columns = ['ion_' + col for col in ion_renamed.columns]
    print(f"Ion columns renamed: {list(ion_renamed.columns)}")
    
    # Perform the merge on serial_number = serialNumber and part_number = partNumber
    print("Merging dataframes...")
    merged_df = pd.merge(
        tipqa_renamed, 
        ion_renamed, 
        left_on=['tipqa_serial_number', 'tipqa_part_number'], 
        right_on=['ion_serialNumber', 'ion_part_partNumber'], 
        how='outer'  # Keep all records from both sides
    )
    
    print(f"Merged dataframe shape: {merged_df.shape}")
    print(f"Total records: {len(merged_df):,}")
    
    # Show breakdown of matches
    tipqa_matches = merged_df['tipqa_serial_number'].notna().sum()
    ion_matches = merged_df['ion_serialNumber'].notna().sum()
    both_matches = merged_df['tipqa_serial_number'].notna() & merged_df['ion_serialNumber'].notna()
    both_matches_count = both_matches.sum()
    
    print(f"\nMatch breakdown:")
    print(f"  - TipQA records: {tipqa_matches:,}")
    print(f"  - Ion records: {ion_matches:,}")
    print(f"  - Records in both: {both_matches_count:,}")
    print(f"  - TipQA only: {tipqa_matches - both_matches_count:,}")
    print(f"  - Ion only: {ion_matches - both_matches_count:,}")
    
    # Display the merged dataframe
    merged_df
    
else:
    print("One or both dataframes are missing")
    if 'tipqa_df' not in locals():
        print("  - tipqa_df is missing")
    if 'combined_df1' not in locals():
        print("  - combined_df1 is missing")
    print("\nPlease run the previous cells to create both dataframes first.")

=== APPENDING TIPQA AND ION DATAFRAMES ===
tipqa_df shape: (13743, 12)
combined_df1 shape: (12795, 45)
Renaming tipqa columns...
TipQA columns renamed: ['tipqa_serial_number', 'tipqa_part_number', 'tipqa_description', 'tipqa_revision', 'tipqa_service_interval_seconds', 'tipqa_asset_type', 'tipqa_location', 'tipqa_last_maintenance_date', 'tipqa_asset_serial_number', 'tipqa_manufacturer', 'tipqa_maintenance_status', 'tipqa_revision_status']
Renaming ion columns...
Ion columns renamed: ['ion_id', 'ion_serialNumber', 'ion__etag', 'ion_status', 'ion_unavailable', 'ion_lotNumber', 'ion_lastMaintainedDate', 'ion_location_id', 'ion_location_name', 'ion_attr_Expiration Date', 'ion_attr_QMS Verified', 'ion_attr_Storage Location', 'ion_attr_DMD-###', 'ion_attr_ion_mirror_id', 'ion_attr_Pending Data', 'ion_attr_data', "ion_attr_counted - July '24", 'ion_attr_Routing Location', 'ion_attr_Joby PO', 'ion_attr_Quality Validation', 'ion_attr_Intended Aircraft', 'ion_attr_RI Number', 'ion_attr_QMS code'

In [121]:
# Sort by tipqa_serial_number first, then ion_serialNumber
merged_df.sort_values(by=['tipqa_serial_number', 'ion_serialNumber'], inplace=True, na_position='last')
merged_df

,tipqa_serial_number,tipqa_part_number,tipqa_description,tipqa_revision,tipqa_service_interval_seconds,tipqa_asset_type,tipqa_location,tipqa_last_maintenance_date,tipqa_asset_serial_number,tipqa_manufacturer,...,ion_part_maintenanceIntervalSeconds,ion_part_attr_Asset Type,ion_part_attr_Primary Location,ion_part_attr_ Used to Make,ion_part_attr_Blocks install with open issues,ion_part_attr_QMS Code,ion_part_attr_Manufacturing Class,ion_part_attr_Responsible ME,ion_abom_id,ion_abom_buildRequirement_id
812,J000-CL01,None,Temporary Cooler,-,0.0,MTE,SRU02,NaT,None,None,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
813,J000-CL02,None,Temporary Cooler,-,0.0,MTE,None,NaT,None,None,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
814,J000-FR01,None,Temporary Freezer,-,0.0,MTE,SRU02,NaT,None,None,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
815,J000-FR02,None,Temporary Freezer,-,0.0,MTE,None,NaT,None,None,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
816,J010-0001,2260B-800-1,"Power Supply, 0-8V/0-1.44A/360W",-,34187400.0,MTE,SRU02,2019-12-19,1407315,KEITHLEY,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
15881,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
15882,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
15883,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN


In [122]:
# Add action_in_ion and reason columns to the front of merged_df
import pandas as pd

print("Adding action_in_ion and reason columns to the front...")

# Create the new columns
merged_df.insert(0, 'reason', '')
merged_df.insert(0, 'action_in_ion', '')

print(f"New shape: {merged_df.shape}")
print(f"First 5 columns: {list(merged_df.columns[:5])}")

# Display the updated dataframe
merged_df

Adding action_in_ion and reason columns to the front...
New shape: (15885, 59)
First 5 columns: ['action_in_ion', 'reason', 'tipqa_serial_number', 'tipqa_part_number', 'tipqa_description']


,action_in_ion,reason,tipqa_serial_number,tipqa_part_number,tipqa_description,tipqa_revision,tipqa_service_interval_seconds,tipqa_asset_type,tipqa_location,tipqa_last_maintenance_date,...,ion_part_maintenanceIntervalSeconds,ion_part_attr_Asset Type,ion_part_attr_Primary Location,ion_part_attr_ Used to Make,ion_part_attr_Blocks install with open issues,ion_part_attr_QMS Code,ion_part_attr_Manufacturing Class,ion_part_attr_Responsible ME,ion_abom_id,ion_abom_buildRequirement_id
812,,,J000-CL01,None,Temporary Cooler,-,0.0,MTE,SRU02,NaT,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
813,,,J000-CL02,None,Temporary Cooler,-,0.0,MTE,None,NaT,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
814,,,J000-FR01,None,Temporary Freezer,-,0.0,MTE,SRU02,NaT,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
815,,,J000-FR02,None,Temporary Freezer,-,0.0,MTE,None,NaT,...,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
816,,,J010-0001,2260B-800-1,"Power Supply, 0-8V/0-1.44A/360W",-,34187400.0,MTE,SRU02,2019-12-19,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15880,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
15881,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,<NA>,None,None,NaN,None,None,None,NaN,None,NaN
15882,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN
15883,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,31557600,MTE,None,NaN,None,None,None,NaN,None,NaN


In [124]:
# Show all columns in merged_df
print("All columns in merged_df:")
print(f"Total columns: {len(merged_df.columns)}")

for i, col in enumerate(merged_df.columns, 1):
    print(f"{i:3d}. {col}")

print(f"\nDataFrame shape: {merged_df.shape}")

All columns in merged_df:
Total columns: 59
  1. action_in_ion
  2. reason
  3. tipqa_serial_number
  4. tipqa_part_number
  5. tipqa_description
  6. tipqa_revision
  7. tipqa_service_interval_seconds
  8. tipqa_asset_type
  9. tipqa_location
 10. tipqa_last_maintenance_date
 11. tipqa_asset_serial_number
 12. tipqa_manufacturer
 13. tipqa_maintenance_status
 14. tipqa_revision_status
 15. ion_id
 16. ion_serialNumber
 17. ion__etag
 18. ion_status
 19. ion_unavailable
 20. ion_lotNumber
 21. ion_lastMaintainedDate
 22. ion_location_id
 23. ion_location_name
 24. ion_attr_Expiration Date
 25. ion_attr_QMS Verified
 26. ion_attr_Storage Location
 27. ion_attr_DMD-###
 28. ion_attr_ion_mirror_id
 29. ion_attr_Pending Data
 30. ion_attr_data
 31. ion_attr_counted - July '24
 32. ion_attr_Routing Location
 33. ion_attr_Joby PO
 34. ion_attr_Quality Validation
 35. ion_attr_Intended Aircraft
 36. ion_attr_RI Number
 37. ion_attr_QMS code
 38. ion_attr_DAR Inspected
 39. ion_attr_Intended U

In [ ]:
# Keep only the specified columns and drop the rest
columns_to_keep = [
    'action_in_ion',
    'reason',
    'tipqa_serial_number',
    'tipqa_part_number',
    'tipqa_description',
    'tipqa_revision',
    'tipqa_service_interval_seconds',
    'tipqa_asset_type',
    'tipqa_location',
    'tipqa_last_maintenance_date',
    'tipqa_asset_serial_number',
    'tipqa_manufacturer',
    'tipqa_maintenance_status',
    'tipqa_revision_status',
    'ion_id',
    'ion_serialNumber',
    'ion__etag',
    'ion_status',
    'ion_lastMaintainedDate',
    'ion_location_id',
    'ion_location_name',
    'ion_attr_Manufacturer',
    'ion_attr_Asset Serial Number',
    'ion_part__etag',
    'ion_part_description',
    'ion_part_id',
    'ion_part_partNumber',
    'ion_part_trackingType',
    'ion_part_partType',
    'ion_part_revision',
    'ion_part_maintenanceIntervalSeconds',
    'ion_part_attr_Asset Type',
    'ion_abom_id',
    'ion_abom_buildRequirement_id'
]

print(f"Original shape: {merged_df.shape}")
print(f"Columns to keep: {len(columns_to_keep)}")

# Keep only the specified columns
merged_df = merged_df[columns_to_keep]

print(f"New shape: {merged_df.shape}")
print(f"Columns dropped: {61 - len(columns_to_keep)}")

# Display the cleaned dataframe
merged_df

Original shape: (15885, 59)
Columns to keep: 34
New shape: (15885, 34)
Columns dropped: 27


,action_in_ion,reason,tipqa_serial_number,tipqa_part_number,tipqa_description,tipqa_revision,tipqa_service_interval_seconds,tipqa_asset_type,tipqa_location,tipqa_last_maintenance_date,...,ion_part_description,ion_part_id,ion_part_partNumber,ion_part_trackingType,ion_part_partType,ion_part_revision,ion_part_maintenanceIntervalSeconds,ion_part_attr_Asset Type,ion_abom_id,ion_abom_buildRequirement_id
812,,,J000-CL01,None,Temporary Cooler,-,0.0,MTE,SRU02,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
813,,,J000-CL02,None,Temporary Cooler,-,0.0,MTE,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
814,,,J000-FR01,None,Temporary Freezer,-,0.0,MTE,SRU02,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
815,,,J000-FR02,None,Temporary Freezer,-,0.0,MTE,None,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
816,SKIP,inactive_unavailable,J010-0001,2260B-800-1,"Power Supply, 0-8V/0-1.44A/360W",-,34187400.0,MTE,SRU02,2019-12-19,...,Multi-Range DC Power Supply,33631.0,2260B-800-1,SERIAL,TOOL,-,31557600,MTE,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15880,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,"Backshell Tool: 2M.514, 5.6mm, 45 Degree, Mold...",3038.0,T-JPS00089-M026-M1-101,SERIAL,TOOL,A,<NA>,None,None,NaN
15881,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,"Backshell Tool: 2M.514, 5.6mm, 45 Degree, Mold...",3039.0,T-JPS00089-M026-M1-102,SERIAL,TOOL,A,<NA>,None,None,NaN
15882,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,Gage,56472.0,209-935,SERIAL,TOOL,-,31557600,MTE,None,NaN
15883,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,Gage,56473.0,209-936,SERIAL,TOOL,-,31557600,MTE,None,NaN
